# T1-11 Dataset + what is frozen

**Time:** 09:00–10:30 (architecture briefing 09:00–09:15) &nbsp;|&nbsp; **Evidence:** dataset report + canary list in `logs/`

**Notebook learning outcomes**

- Shade the architecture card: **frozen** base vs a small **adapter**.
- Split **burst** training from **standing** inference (N11 / N7).
- Inspect an SFT JSONL: instruction, input, output, **train vs held-out**, **canaries**.

You do **not** train today. There is no GPU job in this notebook.


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Overnight readers — two pictures before any rows

Day 1 architecture card (keep it):

```
input text → tokenizer → embedding → transformer blocks → LM head → next token
```

**Training** updates weights. **Inference** uses frozen weights. Yesterday RAG did **not** update weights. It copied a paragraph into the prompt.

**Fine-tuning** (the thing people say when they mean “make it ours”) usually does **not** rewrite all of OLMo. A **LoRA adapter** is a small extra file sitting on some of the blocks. The base stays frozen. You can disable the file. That is rollback.

![Frozen base vs a small adapter](images/t3_frozen_vs_adapter.jpg)

**Burst vs standing** is the cost card from T1-03, said as architecture:

| | Burst | Standing |
|---|---|---|
| What | A training (or LoRA) **job** | The chat **API** that stays up |
| When | Hours, then stop the extra GPU | All day on `:8000` / `:8001` |
| Who asked | CPO “train on judgments” | CPO / TTDF / Finance “own the door” |
| Fail | Buying one box “for AI” with no split | Assuming a LoRA replaces the API |

![Burst training vs standing inference](images/t3_burst_vs_standing.jpg)

**Extra words:**

- **SFT** — supervised fine-tune: each row is instruction + input + desired output.
- **JSONL** — one JSON object per line. That is `data/sft_rows.jsonl`.
- **Held-out** — rows you do **not** train on, so you can see whether the phrasing generalises.
- **Canary** — planted PII. Must not be in a real train split. If the adapter later quotes it, the set was dirty.
- **Licence inheritance** — the adapter is licensed as the **base** plus **every row**. T1-12b.


## Ministry link

| EOI type | What to look at in the rows |
|---|---|
| **Labour / Health** | FT-01, FT-02, FT-07: C2 phrasing, quote, do not hire |
| **all PII papers** | FT-08, FT-09 canaries (Maya Reid, burgess) |
| **Trade** | FT-04: do not reset a password |
| **Defence OPR** | FT-06: do not award |
| **Defence assets** | FT-10: you are not the register |
| **CPO / TTDF / Finance** | Burst vs standing on the card — not a judgment dump in this file |

This lab is **N1, N5, N12** (open-weight base; adapter is a derivative).


## 1. List the rows


In [ ]:
# [RUN]
import civicdesk_ft as ft

print("CivicDesk SFT rows (fiction). Do not replace with ministry judgments.\n")
ft.print_rows()


### Teaching note

`split: train` is what a real job would see. `split: held-out` (FT-07) is a C2 variant you keep back. `canary: true` rows are **poison** if they enter training — they exist so you practise spotting them.


## 2. Dataset report — what stays frozen


In [ ]:
# [RUN]
import json
import civicdesk_ft as ft

report = ft.dataset_report()
print()
print(json.dumps(report, indent=2))
ft.save_log("T1-11_report", report)


### After you run

You should see canaries **FT-08** and **FT-09** named in the printout. `frozen: true` is the classroom rule: OLMo on disk is the same file as Day 1.

**Write-up:** which row would you **delete** before a real job, and which row is a fair C2 example?


## 3. Open one criteria row

Labour/Health shape: match text to C2, quote, do not hire.


In [ ]:
# [CHANGE] try FT-02 (meets C2) or FT-07 (held-out).
ROW_ID = "FT-01"

import json
import civicdesk_ft as ft

row = {r["id"]: r for r in ft.load_rows()}[ROW_ID]
print(json.dumps(row, indent=2))


Look at `output`. It is a **script**, not a hire. The adapter, if it learned this, would only be copying a voice. The vacancy handbook can still change — that text still wants **RAG** (Day 2). Fine-tune does not replace HB-VAC-C2.


In [ ]:
# [OPTIONAL] token-ish length of outputs — a stand-in for 'will this fit'
import civicdesk_ft as ft

for r in ft.load_rows():
    n = len(r["output"].split())
    print(f"{r['id']}  {n:3} words  canary={r['canary']}")
